# Smart Crop Pipeline → Manual Annotate on Roboflow## Bradford Bulls — AI Sponsorship Exposure Valuation### Pipeline```Video → L1 sample (1 FPS) → L2 YOLOv8 person detect + crop      → L3 team filter (Bradford only) → L4 quality filter      → L5 de-dup → L6 top N → Upload to Roboflow      → MANUAL ANNOTATE on Roboflow → Train on Roboflow```> No auto-annotation. High-quality crops only. Human labels = best accuracy.> Runs on **Google Colab GPU**. All data on **Google Drive**.> After Cell 1 (installs), **restart runtime** before Cell 2+.

## 1. Install Dependencies> Run this cell, then **Runtime → Restart runtime**.

In [ ]:
# ============================================================
# ALL INSTALLS — Run once, then RESTART RUNTIME
# ============================================================
!pip uninstall -y autodistill autodistill-grounding-dino sentence-transformers 2>/dev/null || true
!pip install -q ultralytics yt-dlp imagehash scikit-image scikit-learn pillow opencv-python-headless
!pip install -q roboflow

print("\n✅ Installed. NOW RESTART RUNTIME (Runtime → Restart runtime)!")

## 2. Configuration> **TEST MODE** is ON by default — processes only 500 frames, selects 10 crops.> Set `TEST_MODE = False` for full production run after verifying pipeline works.

In [ ]:
# ============================================================
# ⚠️ MASTER CONFIG — All tuneable values in one place
# ============================================================

# --- VIDEO ---
VIDEO_URL = "https://www.youtube.com/watch?v=Yly5ELzUmbw"

# --- SAMPLING ---
TARGET_FPS = 1              # Frames per second to sample

# --- DETECTION ---
YOLO_BATCH = 16             # GPU batch size (lower to 8 if OOM)
MIN_PLAYER_HEIGHT = 120     # Min crop height px
MIN_AREA_RATIO = 0.015      # Min player area as % of frame
CROP_PADDING = 0.05         # 5% padding around bbox

# --- SELECTION ---
MIN_QUALITY = 0.25          # L4: min quality score
DEDUP_HASH_THRESH = 8       # L5: min pHash distance
TIME_BUCKET_SEC = 30        # L6: time diversity bucket (seconds)

# --- ROBOFLOW ---
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # ← Replace with your key
PROJECT_NAME = "kit-sponsor-logos"

# --- CLASS NAMES (for Roboflow project) ---
CLASS_NAMES = [
    "aon_red", "aon_white", "atm_hospitality", "cch_black", "cch_white",
    "chadlaw", "em_workwear", "fairway_flooring", "klg", "mcp_away",
    "mcp_home", "mna_cladding", "mna_support", "paints_lacquers_yellow",
    "top_notch", "bartercard", "floor_tonic", "paints_lacquers_red",
    "romantica_white", "romantica_black", "acs_group",
]

# ============================================================
# ⚠️ TEST vs PRODUCTION
# ============================================================
TEST_MODE = True  # ← Set False for full run

if TEST_MODE:
    MAX_FRAMES_TO_PROCESS = 500
    TARGET_CROPS = 10
    print("🧪 TEST MODE: 500 frames, 10 crops")
else:
    MAX_FRAMES_TO_PROCESS = None
    TARGET_CROPS = 300
    print("🚀 PRODUCTION: full video, 300 crops")

print(f"Target FPS: {TARGET_FPS} | YOLO batch: {YOLO_BATCH} | Target crops: {TARGET_CROPS}")

## 3. Setup

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ No GPU! Runtime → Change runtime type → GPU")

from google.colab import drive
drive.mount("/content/drive")

import os, shutil, csv
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import imagehash
from PIL import Image
from tqdm import tqdm
from sklearn.cluster import KMeans
from ultralytics import YOLO

# All paths on Google Drive
DRIVE_ROOT = Path("/content/drive/MyDrive/Bradford_Bulls")
VIDEOS_DIR     = DRIVE_ROOT / "videos"
BEST_CROPS_DIR = DRIVE_ROOT / "best_crops"
METADATA_DIR   = DRIVE_ROOT / "metadata"

for d in [VIDEOS_DIR, BEST_CROPS_DIR, METADATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("\nPaths (Google Drive):")
for d in [VIDEOS_DIR, BEST_CROPS_DIR, METADATA_DIR]:
    print(f"  {d}")

## 4. Download / Load Video

In [ ]:
existing = sorted(VIDEOS_DIR.glob("*.mp4"), key=lambda f: f.stat().st_mtime, reverse=True)
if existing:
    VIDEO_PATH = existing[0]
    print(f"Found on Drive: {VIDEO_PATH.name} — skipping download")
else:
    print("Downloading...")
    !yt-dlp -f "bestvideo[height<=1080]+bestaudio/best[height<=1080]/best" \
            --merge-output-format mp4 \
            -o "{VIDEOS_DIR}/%(title)s.%(ext)s" \
            --no-playlist "{VIDEO_URL}"
    VIDEO_PATH = sorted(VIDEOS_DIR.glob("*.mp4"), key=lambda f: f.stat().st_mtime, reverse=True)[0]

cap = cv2.VideoCapture(str(VIDEO_PATH))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION = TOTAL_FRAMES / FPS
cap.release()

frame_interval = max(1, int(FPS / TARGET_FPS))
print(f"\n{VIDEO_PATH.name}")
print(f"{DURATION/60:.1f} min | {W}x{H} | {FPS:.0f} FPS | {TOTAL_FRAMES:,} frames")
print(f"Sample every {frame_interval} frames")
if TEST_MODE and MAX_FRAMES_TO_PROCESS:
    print(f"🧪 TEST: stop after {MAX_FRAMES_TO_PROCESS} frames")

## 5. Helper Functions

In [ ]:
def get_jersey_region(crop):
    h, w = crop.shape[:2]
    return crop[int(h*0.15):int(h*0.60), :]

def compute_sharpness(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if gray.size == 0:
        return 0.0
    lap = cv2.Laplacian(gray, cv2.CV_64F).var()
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    ten = np.mean(gx**2 + gy**2)
    mean = gray.mean()
    nvar = gray.var() / (mean + 1e-6)
    return min(lap/300,1)*0.40 + min(ten/5000,1)*0.35 + min(nvar/50,1)*0.25

def compute_frontality(bbox_w, bbox_h):
    if bbox_w == 0: return 0.0
    aspect = bbox_h / bbox_w
    if aspect >= 2.5: return 1.0
    elif aspect >= 1.8: return 0.7
    elif aspect >= 1.3: return 0.4
    else: return 0.2

def get_dominant_colors(image, k=3):
    pixels = image.reshape(-1, 3).astype(np.float32)
    if len(pixels) < k: return pixels[:1]
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    counts = np.bincount(labels.flatten())
    return centers[np.argsort(-counts)].astype(np.uint8)

def fmt_ts(sec):
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}h{m:02d}m{s:02d}s" if h > 0 else f"{m:02d}m{s:02d}s"

print("Helpers loaded.")

## 6. [L1+L2] Sample + Detect + Crop> Sample at 1 FPS → YOLOv8 person detection → crop + score quality.> Crops NOT saved yet — metadata only.

In [ ]:
model = YOLO("yolov8m.pt")
print(f"YOLOv8m | Batch: {YOLO_BATCH}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
frame_area = W * H
all_crops_meta = []
jersey_colors = []
crop_id = 0
batch_frames = []
batch_frame_nums = []

frames_to_process = MAX_FRAMES_TO_PROCESS if TEST_MODE else TOTAL_FRAMES
pbar = tqdm(total=frames_to_process, desc="L1+L2", unit="fr")

frame_num = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if TEST_MODE and MAX_FRAMES_TO_PROCESS and frame_num >= MAX_FRAMES_TO_PROCESS:
        break
    pbar.update(1)

    if frame_num % frame_interval != 0:
        frame_num += 1
        continue

    batch_frames.append(frame)
    batch_frame_nums.append(frame_num)

    if len(batch_frames) < YOLO_BATCH:
        frame_num += 1
        continue

    results = model.predict(batch_frames, classes=[0], conf=0.4, device=device, verbose=False)

    for fn, fr, res in zip(batch_frame_nums, batch_frames, results):
        if res.boxes is None or len(res.boxes) == 0:
            continue
        timestamp = fn / FPS
        for i in range(len(res.boxes)):
            x1, y1, x2, y2 = res.boxes.xyxy[i].cpu().numpy().astype(int)
            conf = float(res.boxes.conf[i].cpu())
            bw, bh = x2 - x1, y2 - y1
            area_ratio = (bw * bh) / frame_area
            if bh < MIN_PLAYER_HEIGHT or area_ratio < MIN_AREA_RATIO:
                continue
            pad_x, pad_y = int(bw * CROP_PADDING), int(bh * CROP_PADDING)
            cx1, cy1 = max(0, x1-pad_x), max(0, y1-pad_y)
            cx2, cy2 = min(W, x2+pad_x), min(H, y2+pad_y)
            crop = fr[cy1:cy2, cx1:cx2]
            if crop.size == 0: continue
            jersey = get_jersey_region(crop)
            if jersey.size == 0: continue
            sharpness = compute_sharpness(jersey)
            front = compute_frontality(bw, bh)
            size_score = min(area_ratio / 0.20, 1.0)
            quality = size_score*0.30 + sharpness*0.30 + front*0.20 + conf*0.20
            jersey_hsv = cv2.cvtColor(jersey, cv2.COLOR_BGR2HSV)
            dom_colors = get_dominant_colors(jersey_hsv, k=2)
            crop_id += 1
            all_crops_meta.append({
                "crop_id": crop_id, "frame_num": fn,
                "timestamp_sec": round(timestamp, 2), "timestamp_hms": fmt_ts(timestamp),
                "bbox": [int(x1), int(y1), int(x2), int(y2)],
                "crop_bbox": [cx1, cy1, cx2, cy2],
                "area_ratio": round(area_ratio, 4), "size_score": round(size_score, 4),
                "sharpness": round(sharpness, 4), "frontality": round(front, 2),
                "det_conf": round(conf, 3), "quality": round(quality, 4),
                "dom_color_1_hsv": dom_colors[0].tolist(),
                "dom_color_2_hsv": dom_colors[1].tolist() if len(dom_colors) > 1 else dom_colors[0].tolist(),
            })
            jersey_colors.append(dom_colors[0])

    batch_frames, batch_frame_nums = [], []
    frame_num += 1

# Remaining batch
if batch_frames:
    results = model.predict(batch_frames, classes=[0], conf=0.4, device=device, verbose=False)
    for fn, fr, res in zip(batch_frame_nums, batch_frames, results):
        if res.boxes is None or len(res.boxes) == 0: continue
        timestamp = fn / FPS
        for i in range(len(res.boxes)):
            x1, y1, x2, y2 = res.boxes.xyxy[i].cpu().numpy().astype(int)
            conf = float(res.boxes.conf[i].cpu())
            bw, bh = x2-x1, y2-y1
            area_ratio = (bw*bh) / frame_area
            if bh < MIN_PLAYER_HEIGHT or area_ratio < MIN_AREA_RATIO: continue
            pad_x, pad_y = int(bw*CROP_PADDING), int(bh*CROP_PADDING)
            cx1, cy1 = max(0, x1-pad_x), max(0, y1-pad_y)
            cx2, cy2 = min(W, x2+pad_x), min(H, y2+pad_y)
            crop = fr[cy1:cy2, cx1:cx2]
            if crop.size == 0: continue
            jersey = get_jersey_region(crop)
            if jersey.size == 0: continue
            sharpness = compute_sharpness(jersey)
            front = compute_frontality(bw, bh)
            size_score = min(area_ratio / 0.20, 1.0)
            quality = size_score*0.30 + sharpness*0.30 + front*0.20 + conf*0.20
            jersey_hsv = cv2.cvtColor(jersey, cv2.COLOR_BGR2HSV)
            dom_colors = get_dominant_colors(jersey_hsv, k=2)
            crop_id += 1
            all_crops_meta.append({
                "crop_id": crop_id, "frame_num": fn,
                "timestamp_sec": round(timestamp, 2), "timestamp_hms": fmt_ts(timestamp),
                "bbox": [int(x1), int(y1), int(x2), int(y2)],
                "crop_bbox": [cx1, cy1, cx2, cy2],
                "area_ratio": round(area_ratio, 4), "size_score": round(size_score, 4),
                "sharpness": round(sharpness, 4), "frontality": round(front, 2),
                "det_conf": round(conf, 3), "quality": round(quality, 4),
                "dom_color_1_hsv": dom_colors[0].tolist(),
                "dom_color_2_hsv": dom_colors[1].tolist() if len(dom_colors) > 1 else dom_colors[0].tolist(),
            })
            jersey_colors.append(dom_colors[0])

pbar.close()
cap.release()
jersey_colors = np.array(jersey_colors) if jersey_colors else np.array([]).reshape(0,3)
print(f"\nL1+L2: {len(all_crops_meta):,} player crops")

## 7. [L3] Team Classification> Cluster jersey colors → pick Bradford Bulls cluster.

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(jersey_colors.astype(np.float32))

for i, meta in enumerate(all_crops_meta):
    meta["team_cluster"] = int(cluster_labels[i])

c0 = sum(1 for c in cluster_labels if c == 0)
c1 = sum(1 for c in cluster_labels if c == 1)
print(f"Cluster 0: {c0:,} | Cluster 1: {c1:,}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for cid in [0, 1]:
    samples = sorted([m for m in all_crops_meta if m["team_cluster"]==cid], key=lambda x: x["quality"], reverse=True)[:6]
    for j, meta in enumerate(samples):
        cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
        ret, frame = cap.read()
        if not ret: continue
        cx1, cy1, cx2, cy2 = meta["crop_bbox"]
        axes[cid][j].imshow(cv2.cvtColor(frame[cy1:cy2, cx1:cx2], cv2.COLOR_BGR2RGB))
        axes[cid][j].set_title(f"Q={meta['quality']:.2f}", fontsize=9)
        axes[cid][j].axis("off")
    axes[cid][0].set_ylabel(f"Cluster {cid}\n({[c0,c1][cid]})", fontsize=11, fontweight="bold")
cap.release()
plt.suptitle("Which cluster is BRADFORD BULLS?", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ⚠️ SET THIS: Which cluster is Bradford Bulls? (0 or 1)
BRADFORD_CLUSTER = 0  # ← CHANGE after looking at images above

bradford_crops = [m for m in all_crops_meta if m["team_cluster"] == BRADFORD_CLUSTER]
print(f"Bradford: {len(bradford_crops):,} crops (removed {len(all_crops_meta)-len(bradford_crops):,})")

## 8. [L4+L5+L6] Quality → De-dup → Top N

In [ ]:
quality_filtered = sorted([m for m in bradford_crops if m["quality"] >= MIN_QUALITY],
                         key=lambda x: x["quality"], reverse=True)
print(f"L4 quality (>= {MIN_QUALITY}): {len(quality_filtered):,}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
selected = []
selected_hashes = []
time_bucket_counts = Counter()
MAX_PER_BUCKET = max(3, TARGET_CROPS // (int(DURATION / TIME_BUCKET_SEC) + 1) + 2)

for meta in tqdm(quality_filtered, desc="L5+L6"):
    bucket = int(meta["timestamp_sec"] // TIME_BUCKET_SEC)
    if time_bucket_counts[bucket] >= MAX_PER_BUCKET: continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret: continue
    cx1, cy1, cx2, cy2 = meta["crop_bbox"]
    crop = frame[cy1:cy2, cx1:cx2]
    if crop.size == 0: continue

    crop_hash = imagehash.phash(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
    if any(crop_hash - h < DEDUP_HASH_THRESH for h in selected_hashes): continue

    meta["crop_hash"] = str(crop_hash)
    selected.append(meta)
    selected_hashes.append(crop_hash)
    time_bucket_counts[bucket] += 1
    if len(selected) >= TARGET_CROPS: break

cap.release()
print(f"\nSelected: {len(selected)} crops")
if selected:
    quals = [s["quality"] for s in selected]
    print(f"Quality: {min(quals):.3f} — {max(quals):.3f} (mean {np.mean(quals):.3f})")

## 9. Save Crops to Google Drive

In [ ]:
selected.sort(key=lambda x: x["timestamp_sec"])
cap = cv2.VideoCapture(str(VIDEO_PATH))
metadata_rows = []

for idx, meta in enumerate(tqdm(selected, desc="Saving crops")):
    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret: continue
    cx1, cy1, cx2, cy2 = meta["crop_bbox"]
    crop = frame[cy1:cy2, cx1:cx2]

    filename = f"crop_{idx+1:04d}_{meta['timestamp_hms']}_q{meta['quality']:.2f}.jpg"
    cv2.imwrite(str(BEST_CROPS_DIR / filename), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
    meta["filename"] = filename
    metadata_rows.append({
        "crop_id": idx+1, "filename": filename,
        "frame_num": meta["frame_num"], "timestamp_sec": meta["timestamp_sec"],
        "timestamp_hms": meta["timestamp_hms"], "quality": meta["quality"],
        "sharpness": meta["sharpness"], "frontality": meta["frontality"],
        "area_pct": round(meta["area_ratio"]*100, 2),
        "source_video": VIDEO_PATH.name,
    })

cap.release()
df = pd.DataFrame(metadata_rows)
csv_path = METADATA_DIR / "selected_crops_index.csv"
df.to_csv(csv_path, index=False)

print(f"\nSaved {len(metadata_rows)} crops:")
print(f"  Crops: {BEST_CROPS_DIR}")
print(f"  CSV:   {csv_path}")

## 10. Preview

In [ ]:
crop_files = sorted(BEST_CROPS_DIR.glob("crop_*.jpg"))
get_q = lambda p: float(p.stem.split("_q")[-1])
by_quality = sorted(crop_files, key=get_q, reverse=True)

n = min(12, len(by_quality))
cols = min(4, n)
rows = max(1, (n + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 4*rows))
axes = np.array(axes).flatten() if n > 1 else [axes]

for i, fp in enumerate(by_quality[:n]):
    img = cv2.imread(str(fp))
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Q={get_q(fp):.2f} | {fp.stem.split('_')[2]}", fontsize=9)
    axes[i].axis("off")
for j in range(n, len(axes)): axes[j].axis("off")

plt.suptitle(f"Top {n} Crops — Ready for manual annotation", fontsize=13)
plt.tight_layout()
plt.show()
print(f"Total: {len(crop_files)} crops")

## 11. Upload to Roboflow (No Annotations)> Crops are uploaded **without labels**. You annotate manually on Roboflow.>> **After upload:**> 1. Go to Roboflow → Annotate tab> 2. Draw bounding boxes around each logo> 3. Assign class from the 21 classes below> 4. Generate dataset version → Train on Roboflow (or export YOLO format)

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace()

try:
    project = workspace.project(PROJECT_NAME)
    print(f"Connected: {project.name}")
except Exception:
    project = workspace.create_project(
        project_name="Kit Sponsor Logos",
        project_type="object-detection",
        project_license="MIT",
        annotation="kit-sponsor-logos",
    )
    print("Created new project: Kit Sponsor Logos")

# Upload crops WITHOUT annotations
crop_files = sorted(BEST_CROPS_DIR.glob("crop_*.jpg"))
success, errors = 0, 0

for img_path in tqdm(crop_files, desc="Uploading to Roboflow"):
    try:
        project.upload(image_path=str(img_path), split="train")
        success += 1
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"  Error: {e}")

print(f"\nUploaded! Success: {success} | Errors: {errors}")
print(f"\n📋 Now go annotate manually:")
print(f"   https://app.roboflow.com/{workspace.url}/{PROJECT_NAME}/annotate")
print(f"\n21 classes to use:")
for i, c in enumerate(CLASS_NAMES):
    print(f"   {i:2d}: {c}")

## 12. Next Steps### On Roboflow:1. **Annotate** — Draw bounding boxes around each logo, assign correct class2. **Generate** — Create dataset version (apply preprocessing + augmentation)3. **Train** — Use Roboflow Train (YOLOv8) or export dataset to train locally### 21 Logo Classes:| ID | Code Name | Logo ||----|-----------|------|| 0 | aon_red | Aon (Red) || 1 | aon_white | Aon (White) || 2 | atm_hospitality | ATM Hospitality || 3 | cch_black | CCH (Black) || 4 | cch_white | CCH (White) || 5 | chadlaw | ChadLaw || 6 | em_workwear | EM Workwear || 7 | fairway_flooring | Fairway Flooring || 8 | klg | KLG || 9 | mcp_away | MCP (Away) || 10 | mcp_home | MCP (Home) || 11 | mna_cladding | MNA Cladding || 12 | mna_support | MNA Support || 13 | paints_lacquers_yellow | Paints & Lacquers (yellow) || 14 | top_notch | Top Notch || 15 | bartercard | Bartercard || 16 | floor_tonic | Floor Tonic || 17 | paints_lacquers_red | Paints & Lacquers (red) || 18 | romantica_white | Romantica (White) || 19 | romantica_black | Romantica (Black) || 20 | acs_group | ACS Group |### Roboflow Train:- Go to **Versions** → **Generate** → choose augmentation- Click **Train** → select YOLOv8 → Start training- Model will be deployed on Roboflow for inference API